## Topic: Introduction of Retrieval-Augmented Generation (RAG)

### Agenda
- 1. Introduction of Retrieval-Augmented Generation (RAG)

- 2. How does RAG Work?

- 3. Key Takeaways

### 1. Introduction of Retrieval-Augmented Generation (RAG)

- Definition:
    - RAG (Retrieval-Augmented Generation) is a technique that allows an LLM to retrieve relevant information from an external knowledge source and use that information to generate an answer.

- In simple terms:
    - RAG = Retrieve relevant information → Give it to the LLM → Generate an answer


- RAG combines two distinct processes:
    - 1. Retrieval: Finding relevant, factual context from an external data source based on the user's prompt.

    - 2. Generation: Passing both the fetched context and original prompt to an LLM to synthesize a grounded, accurate answer.
    

In [ ]:
""" 
- RAG adds an external knowledge retrieval step:
===============================================

        User Question
            ↓
        Retriever
            ↓
        Relevant Information
            ↓
            LLM
            ↓
          Answer


- Key Rule of Thump:
    - Traditional LLM -> Closed-Book Exam
    - RAG System -> Open-Book Exam

"""

In [ ]:
"""
    - Why do we need RAG?

┌─────────────────────────────────────────────────────────────┐
│                 WHY WE NEED RAG                             │
│                                                             │
│  1. LLMS DO NOT KNOW YOUR PRIVATE DATA                      │
│     Company policies, PDFs, Slack messages, databases,      │
│     support tickets, internal wikis, and contracts.         │
│                                                             │
│  2. LLMS MAY HAVE OUTDATED KNOWLEDGE                        │
│     Model training has a knowledge cutoff date.             │
│     RAG can retrieve new documents immediately.             │
│                                                             │
│  3. LLMS CAN HALLUCINATE                                    │
│     Without reliable context, LLMs may invent answers.      │
│     RAG grounds answers in retrieved documents.             │
│                                                             │
│  4. FULL DOCUMENTS DO NOT FIT IN PROMPTS                    │
│     You cannot send 10,000 PDFs to an LLM every time.       │
│     RAG retrieves only the most relevant chunks.            │
│                                                             │
│  5. FINE-TUNING IS NOT ALWAYS THE ANSWER                    │
│     Fine-tuning is expensive and slow for changing facts.   │
│     RAG updates knowledge by re-indexing documents.         │
└─────────────────────────────────────────────────────────────┘



┌────────────────────────────────────────────────────────────────────────┐
│                        THE STATIC LLM DILEMMA                          │
│                                                                        │
│   Pre-trained Weights (Frozen)       Private / Live Enterprise Data    │
│   ┌───────────────────────────┐      ┌─────────────────────────────┐   │
│   │ • Training cutoff date    │  ≠   │ • Internal company PDFs     │   │
│   │ • No private company data │      │ • Live inventory & pricing  │   │
│   │ • Prone to hallucinations │      │ • Real-time API data & logs │   │
│   └───────────────────────────┘      └─────────────────────────────┘   │
└────────────────────────────────────────────────────────────────────────┘


- The Core Motivations for RAG:
    - 1. Dynamic Data Updates: Knowledge bases update constantly without expensive retraining cycles.

    - 2. Data Privacy & Security: Proprietary enterprise documents stay in private storage and are never fed into public model training datasets.

    - 3. Factual Traceability: Responses reference and cite verifiable documents for auditing and compliance.

    - 4. Cost Reduction: Injecting context at runtime is significantly cheaper and faster than fine-tuning or continuous pre-training.
"""

### 2. How does RAG Work?

- A complete RAG system normally has two major phases:
    - 1. INDEXING PHASE  
        - → Prepare/Create External Knowledge Base (context).

    - 2. QUERY PHASE     
        - → Retrieve and answer user questions
        

- Another way we say , the RAG system contain
    - 1. Indexing
        - Prepare/Create External Knowledge Base (context).
    
    - 2. Retrieval
        - understand the user query and go to the knowledge base, fetch the relevant(chunk) context.

    - 3. Augmentation
        - Creating a Prompt by combine the **user query** and **retrieval relevant context**.
    
    - 4. Generation
        - Pass the Prompt (that are created by Augmentation) to LLM and get Ground Response from the LLM.


In [ ]:
""" 

                RAG
                 │
       ┌─────────┴─────────┐
       │                   │
       ▼                   ▼
   INDEXING             RETRIEVAL
   PHASE                  PHASE

"""

#### Phase 1 — Indexing 
- Before users ask questions, we need to prepare our knowledge base.

- Definition of Indexing: 
    - Indexing is the process of preparing our knowledge base so that it can be efficiently searched at query time. 


- This steps consists of 4 sub-steps:
    - 1. Document Ingestion: we load our source knowledge into memory.
        - PDF report, word documents
        - YouTube transcripts, blog pages.
        - GitHub repos, internal wikis
        - SQL records, scraped webpages

        - To do that the LangChain tools:
            - PyPDFLoader, YoutubeLoader, WebBaseLoader, GitLoader and so on..

    - 2. Text Chunking:
        - Break large documents into small, semantically meaningful chunks

        - Why chunk?
            - LLMs have context limit (eg. 4k-32k tokens).
            - smaller chunks are more focused (better semantic search).

        - In LangChain Tools:
            - RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter and so on.

    
    - 3. Embedding Generation:
        - Convert each chunk into a dense vector(embedding) that captures it's meaning.

        - Why embedding?
            - Similar ideas land close together in vector space.
            - allows fast, fuzzy, semantic search

        
        - In LangChain tools:
            - OpenAIEmbeddings, HuggingFaceEmbeddings and so on..

    - 4. Storage in a Vector Store:
        - Store the Vectors along with the original chunk text with metadata in a vector database.

        - Vector DB options:
            - Local: FAISS, Chroma
            - Cloud: Pinecone, Weaviate, Qdrant

        - This Vector Store is our External Knowledge Base.


In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│                    RAG INDEXING PHASE                       │
│                                                             │
│  1. LOAD DOCUMENTS                                          │
│     PDFs, DOCX, CSV, websites, Notion, Slack, databases     │
│                                                             │
│  2. SPLIT DOCUMENTS                                         │
│     Large documents → smaller chunks                        │
│     Example: 1000 characters with 200 overlap               │
│                                                             │
│  3. CREATE EMBEDDINGS                                       │
│     Text chunks → numerical vectors                         │
│     "refund policy" → [0.12, -0.38, 0.91, ...]              │
│                                                             │
│  4. STORE IN VECTOR DATABASE                                │
│     Vectors + text + metadata saved in FAISS/Chroma/etc.    │
│                                                             │
│  OUTPUT: Searchable knowledge base                          │
└─────────────────────────────────────────────────────────────┘

"""

#### Phase 2 — Retrieval
- Now the user asks:
    - How long do refunds take?

    - The RAG system needs to find relevant information.


- Definition Retrieval:
    - Retrieval is the real-time process of finding the most relevant pieces of information from a pre-built index (created during indexing) based on the user’s question.

- It's like asking:
    - From all the knowledge i have with 3-5 chunks are most helpful to answer the query?

- The steps of retrieval:
    - 1. input the user query
    - 2. convert the user query to embedding dense vector with same dimension.
    - 3. Fetch(search) the closest relevant embedding dense vector from the Knowledge base.
    - 4. applying a ranking which embedding dense vector are most important for the user query.

    - 5. Return the most relevant chunks (content) or top k chunks.



In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│                     RAG QUERY PHASE                         │
│                                                             │
│  1. USER ASKS A QUESTION                                    │
│     "How long do refunds take?"                             │
│                                                             │
│  2. RETRIEVER SEARCHES KNOWLEDGE BASE                       │
│     Converts query into embedding vector                    │
│     Finds similar/relevant document chunks                  │
│                                                             │
│  3. RETRIEVED CONTEXT IS ADDED TO PROMPT                    │
│     Context + User Question → LLM prompt                    │
│                                                             │
│  4. LLM GENERATES GROUNDED ANSWER                           │
│     Uses provided context instead of guessing               │
│                                                             │
│  OUTPUT: Answer + optional source citations                 │
└─────────────────────────────────────────────────────────────┘

"""

#### Augmentation

- Definition Augmentation:
    - Augmentation refers to the step where the retrieved documents (chunks of relevant context) are combined with the user’s query to form a new, enriched prompt for the LLM.
    
    - Augmentation is the middle step between retrieval and generation.

    - It is the process of transforming retrieved documents into a well-structured prompt that the LLM can use.


- One-line definition:
    - Augmentation = Formatting + Grounding + Injecting retrieved context into the LLM prompt.


- Key Note:
    - Retrieval     → Find relevant documents
    - Augmentation  → Combine those documents with the question
    - Generation    → LLM produces the final answer


In [ ]:
"""
┌──────────────────────────────────────────────────────────────────┐
│              COMPLETE RAG PIPELINE WITH AUGMENTATION             │
│                                                                  │
│  ──────────────────── INDEXING PHASE (Offline) ────────────────  │
│                                                                  │
│    [Loader] → [Splitter] → [Embeddings] → [Vector DB]            │
│                                                                  │
│  ──────────────────── QUERY PHASE (Online) ────────────────────  │
│                                                                  │
│    User Query                                                    │
│        │                                                         │
│        ▼                                                         │
│    [Retriever]  →  List[Document]                                │
│        │                                                         │
│        ▼                                                         │
│    ┌───────────────────────────────────────────────────────┐    │
│    │             AUGMENTATION LAYER  ← THIS PART           │    │
│    │                                                       │    │
│    │  1. DEDUPLICATE   → remove repeated chunks             │    │
│    │  2. REORDER       → best docs at start & end           │    │
│    │  3. TRIM          → fit token budget                   │    │
│    │  4. FORMAT        → docs → one context string          │    │
│    │  5. TEMPLATE      → instructions + context + question  │    │
│    │                                                       │    │
│    │  Output: Final Augmented Prompt (a single string)      │    │
│    └───────────────────────────────────────────────────────┘    │
│        │                                                         │
│        ▼                                                         │
│    [LLM]  →  Grounded Answer (+ citations)                       │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘



"""

In [ ]:
""" 
Internal Flow of Augmentation:
==================================
┌─────────────────────────────────────────────────────────────┐
│              AUGMENTATION INTERNAL FLOW                     │
│                                                             │
│  1. RECEIVE RETRIEVED DOCUMENTS                             │
│     [Document(page_content="...", metadata={...}), ...]     │
│                                                             │
│  2. CLEAN & DEDUPLICATE                                     │
│     Remove near-duplicate chunks (same source, similar text)│
│                                                             │
│  3. ORDER FOR ATTENTION                                     │
│     Best-ranked docs at START and END                       │
│     Weakest in the MIDDLE (lost-in-the-middle mitigation)   │
│                                                             │
│  4. TRIM TO TOKEN BUDGET                                    │
│     If context > budget → drop lowest-ranked docs           │
│     Reserve space for: input + output + instructions        │
│                                                             │
│  5. FORMAT CONTEXT BLOCK                                    │
│     Wrap each doc in delimiters with source labels          │
│                                                             │
│     <context>                                               │
│       [Doc 1 | source=refund_policy.pdf, page=2]            │
│       Refunds are processed within 5-10 business days...    │
│                                                             │
│       [Doc 2 | source=shipping_policy.pdf]                  │
│       Shipping fees are non-refundable...                   │
│     </context>                                              │
│                                                             │
│  6. BUILD FINAL PROMPT                                      │
│     System Instructions                                     │
│     + Context Block                                         │
│     + User Question                                         │
│     = Augmented Prompt (single string)                      │
│                                                             │
│  7. RETURN AUGMENTED PROMPT → LLM                           │
└─────────────────────────────────────────────────────────────┘


"""

In [ ]:
"""
"""
===
# Example of Augmentation:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.

If the answer is not available in the context, say:
"I don't know based on the available documents."

Context:
{context}

Question:
{question}

Answer:
""")




===
"""
"""

#### Generation

- Definition Generation:
    - Generation is the final step where a Large Language Model (LLM) uses the user’s query and the retrieved & augmented context to generate a response.


In [ ]:
"""
RAG = Three Core Operation:
===========================

       RAG
        │
        ├── 1. RETRIEVE
        │       ↓
        │   Find relevant information
        │
        ├── 2. AUGMENT
        │       ↓
        │   Add information to prompt
        │
        └── 3. GENERATE
                ↓
            LLM generates answer



"""

In [ ]:
"""    - Complete RAG Architecture
        ===============================
        
                    ┌───────────────────┐
                    │   KNOWLEDGE BASE  │
                    │                   │
                    │ PDFs              │
                    │ Websites          │
                    │ Documents         │
                    │ Databases         │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │  DOCUMENT LOADER  │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │  TEXT SPLITTER    │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │ EMBEDDING MODEL   │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │   VECTOR STORE    │
                    │                   │
                    │ Chroma / FAISS /  │
                    │ Qdrant / etc.     │
                    └─────────┬─────────┘
                              │
                              │
══════════════════════════════╪══════════════════════════════
                              │
                         USER QUERY
                              │
                              ▼
                    ┌───────────────────┐
                    │ EMBEDDING MODEL   │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │    RETRIEVER      │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │ RELEVANT CHUNKS   │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │      PROMPT       │
                    │                   │
                    │ Context + Query   │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │       LLM         │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │      ANSWER       │
                    └───────────────────┘

"""

### Key Takeaways

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│               RETRIEVAL-AUGMENTED GENERATION (RAG)               │
│                                                                  │
│  WHAT:  Architecture that retrieves relevant documents and       │
│         gives them to an LLM before it generates an answer.      │
│                                                                  │
│  WHY:   LLMs do not know private, current, or domain-specific    │
│         information reliably.                                    │
│                                                                  │
│  PROBLEM SOLVED:                                                 │
│    - Knowledge cutoff                                            │
│    - Private company data                                        │
│    - Hallucinations                                              │
│    - Limited context windows                                     │
│    - Expensive fine-tuning for changing facts                    │
│                                                                  │
│  TWO PHASES:                                                     │
│    1. Indexing: Load → Split → Embed → Store                     │
│    2. Querying: Query → Retrieve → Prompt → LLM Answer           │
│                                                                  │
│  CORE PIPELINE:                                                  │
│    [Loader] → [Splitter] → [Embeddings] → [Vector DB]            │
│                                                   ↓              │
│    User Query → [Retriever] → Relevant Context → [LLM]           │
│                                                   ↓              │
│                                          Grounded Answer         │
│                                                                  │
│  DEFAULT STARTER STACK:                                          │
│    Loader:      PyPDFLoader / WebBaseLoader                      │
│    Splitter:    RecursiveCharacterTextSplitter(1000, 200)        │
│    Embeddings:  OpenAIEmbeddings or HuggingFaceEmbeddings        │
│    Vector DB:   FAISS (local) / Chroma                           │
│    Retriever:   Similarity Search, k=4                           │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "The LLM can only answer as well as the documents retrieved.    │
│   Improve retrieval quality before over-engineering prompts."    │
└──────────────────────────────────────────────────────────────────┘

"""